# NLP Projekt 1

## Introduction

The goal of this project is to create two models with different architectures to predict the correct answer option of two possibilities for a question or sentence using the Physical Commonsense (PIQA) Dataset.
Another important aspect of this project is the decision making during the implementation. Following steps are included:

- Setup
- Preprocessing
- Data Loading
- Embeddings
- Architectures
- Training
- Evaluation
- Interpretation

[wandb Report](https://wandb.ai/nlp-1/nlp-project1/reports/NLP-Project-one-report--VmlldzoxNjQ0MDQxMQ)

During the implementation were the AI tools Gemini and ChatGPT used for:

- Runtime error analysis and/or fix
- Suggestions for decisions
- Markdown text formatting
- Check if the self made decisions make sense

## Setup

The entire project was developed on the Gpuhub server with the pytorch environment. The python version is 3.13.12.

In [6]:
!pip install datasets torch wandb nltk fasttext
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torch.nn.functional as F

from datasets import load_dataset
import numpy as np
import random
import wandb
import nltk

from nltk.tokenize import word_tokenize
from collections import Counter
import string
import re
import fasttext.util

nltk.download('punkt')
nltk.download("punkt_tab")

# A random seed is set to get the same random sequences
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

#after testing on multiple devices like cpu and mps, cuda was the fastest
DEVICE = torch.device('cuda')

wandb.login()

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jovyan/.netrc.
wandb: Currently logged in as: daniel-polgar (nlp-1) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Data Loading

The dataset loading and the split code is based on the requirement in the project introduction slides from ilias. However the main branch on Huggingface did not work, so with the suggestion of Gemini the branch with parquet file conversion was used.

So the data is split up into:
- training: training except the last 1000 from the training
- validation: last 1000 from training
- test: validation

In [8]:
train_data = load_dataset('ybisk/piqa', revision='refs/convert/parquet', split='train[:-1000]')
valid_data = load_dataset('ybisk/piqa', revision='refs/convert/parquet', split='train[-1000:]')
test_data = load_dataset('ybisk/piqa', revision='refs/convert/parquet', split='validation')

#Data citation
#@inproceedings{Bisk2020,
#  author = {Yonatan Bisk and Rowan Zellers and
#            Ronan Le Bras and Jianfeng Gao
#            and Yejin Choi},
#  title = {PIQA: Reasoning about Physical Commonsense in
#           Natural Language},
#  booktitle = {Thirty-Fourth AAAI Conference on
#               Artificial Intelligence},
#  year = {2020},
#}

In [9]:
train_data

Dataset({
    features: ['goal', 'sol1', 'sol2', 'label'],
    num_rows: 15113
})

In [10]:
valid_data

Dataset({
    features: ['goal', 'sol1', 'sol2', 'label'],
    num_rows: 1000
})

In [11]:
test_data

Dataset({
    features: ['goal', 'sol1', 'sol2', 'label'],
    num_rows: 1838
})

In [12]:
train_data[:10]

{'goal': ["When boiling butter, when it's ready, you can",
  'To permanently attach metal legs to a chair, you can',
  'how do you indent something?',
  'how do you shake something?',
  'Clean tires',
  'how do you taste something?',
  'To create a makeshift ice pack,',
  "What should I use as a stain on a wooden bowl I've just made.",
  'How to boil eggs.',
  'how do you stab something?'],
 'sol1': ['Pour it onto a plate',
  'Weld the metal together to get it to stay firmly in place',
  'leave a space before starting the writing',
  'move it up and down and side to side quickly.',
  'Pour water, cape off caked on dirt. Use  speed wool to clean out crevices and sparrow spaces.',
  'smell it enough to taste it.',
  'take a sponge and soak it in oil. Put the sponge in a refrigerator and let it freeze. Once frozen, take it out and put it in a ziploc bag. You can now use it as an ice pack.',
  'You should coat the wooden bowl with a butcher block oil & finish per manufacturer directions.',

## Preprocessing

For this part of the preprocessing following decisions were made:

**lower case**: all the text was converted to lower case, for consistent tokens

**word tokenization**: since we want to find out the answer based on the meaning of the sentence, it makes sense to get words as meaningful units at the tokenization.

**html cleaning**: the data contains no html tags, so they do not have to be cleaned

**string punctuation strip**: After the word tokenization, there were some tokens like "'word". To avoid those, it has been used.

**lemmatizer**: the chosen embeddig was fasttext, which can recognize the words without lemmatization, so it was not needed.

**processing**: between hugging face map and pytorch Dataset & Dataloader was the second chosen because of the better understanding from the docs [DataSet & DataLoader](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html)

ChatGPT found all these decisions reasonable.

In [13]:
def contains_html(text):
    return bool(re.search(r'<[^>]+>', text))

# word tokenize i lower case with string punctuation strip
def tokenize(text):
    tokens = word_tokenize(text.lower())
    clean_tokens = []
    for token in tokens:
        clean_token = token.strip(string.punctuation)
        clean_tokens.append(clean_token)
    return clean_tokens
    

# Build vocab
counter = Counter()
all_tokens = []

for example in train_data:
    if contains_html(example['goal']) or contains_html(example['sol1']) or contains_html(example['sol1']):
        print("contains html!!!!")
    tokens = tokenize(example['goal']) + tokenize(example['sol1']) + tokenize(example['sol2'])
    counter.update(tokens)
    all_tokens.extend(tokens)

# add all words to the vocabulary starting with index 2 having the most common word
vocab = {word: i+2 for i, (word, _) in enumerate(counter.most_common())}
# chatGPT suggested to add <PAD> and <UNK> to avoid different sequence lengths in a batch and <UNK> for yet unseen words.
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

#print for unknown and special tokens for both training and validation set were created with the help of ChatGPT

print("Vocabulary size:", len(vocab))
special_tokens_train = [t for t in all_tokens if any(c in string.punctuation for c in t)]
print("[TRAIN] Number total of tokens with special characters:", len(special_tokens_train))
print("[TRAIN] Number unique of tokens with special characters:", len(set(special_tokens_train)))
print("[TRAIN] Sample special tokens:", special_tokens_train[:20])

train_unk_tokens = []
train_total_tokens = []

for example in train_data:
    tokens = tokenize(example['goal']) + tokenize(example['sol1']) + tokenize(example['sol2'])
    for t in tokens:
        train_total_tokens.append(t)
        if t not in vocab:
            train_unk_tokens.append(t)

print("[TRAIN] Unknown token count:", len(set(train_unk_tokens)))
print("[TRAIN] Total tokens:", len(train_total_tokens))
print("[TRAIN] UNK ratio:", len(train_unk_tokens) / len(vocab) if len(set(train_total_tokens)) > 0 else 0)

k_tokens_list = []
unk_tokens_list = []
special_tokens_val = []

for example in valid_data:
    if contains_html(example['goal']) or contains_html(example['sol1']) or contains_html(example['sol1']):
        print("contains html!!!!")
    tokens = tokenize(example['goal']) + tokenize(example['sol1']) + tokenize(example['sol2'])
    for t in tokens:
        k_tokens_list.append(t)
        if t not in vocab:
            unk_tokens_list.append(t)
        if any(c in string.punctuation for c in t):
            special_tokens_val.append(t)

print("[VALIDATION] total unknown token count:", len(unk_tokens_list))
print("[VALIDATION] unique unknown token count:", len(set(unk_tokens_list)))
print("[VALIDATION] Total tokens:", len(k_tokens_list))
print("[VALIDATION] UNK ratio:", len(set(unk_tokens_list)) / len(vocab))

print("[VALIDATION] Sample UNK tokens:", list(set(unk_tokens_list))[:20])
print("[VALIDATION] Number total of special tokens:", len(special_tokens_val))
print("[VALIDATION] Number unique of special tokens:", len(set(special_tokens_val)))
print("[VALIDATION] Sample special tokens:", special_tokens_val[:20])

# encode the text with each word using its id from the vocabulary
def encode(text):
    #set max length to 50 to make sure, that every sentence can be hanled in one row
    max_len = 50
    tokens = tokenize(text)
    ids = [vocab.get(t, 1) for t in tokens][:max_len]
    ids += [0] * (max_len - len(ids))
    return ids


class PIQADataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex = self.data[idx]
        goal = encode(ex['goal'])
        sol1 = encode(ex['sol1'])
        sol2 = encode(ex['sol2'])
        label = ex['label']
        return torch.tensor(goal), torch.tensor(sol1), torch.tensor(sol2), torch.tensor(label)

train_dataset = PIQADataset(train_data)
val_dataset = PIQADataset(valid_data)
test_dataset = PIQADataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

Vocabulary size: 15613
[TRAIN] Number total of tokens with special characters: 5450
[TRAIN] Number unique of tokens with special characters: 1188
[TRAIN] Sample special tokens: ['medium-high', 'medium-high', '5/4', '3/4', 'capri-sun', 'low-medium', 'low-medium', "n't", "n't", '1:1', '1:1', '1/2', '1/2', 'post-its', "n't", 'half-way', 'half-way', 'coca-cola', "n't", "n't"]
[TRAIN] Unknown token count: 0
[TRAIN] Total tokens: 748349
[TRAIN] UNK ratio: 0.0
[VALIDATION] total unknown token count: 675
[VALIDATION] unique unknown token count: 429
[VALIDATION] Total tokens: 51210
[VALIDATION] UNK ratio: 0.027477102414654453
[VALIDATION] Sample UNK tokens: ['regained', '16v', 'californians', 'acetic', 'hilmalayean', 'pusing', 'syphon', 'rib', 'contraceptives', 'gamestop', 'vapo', 'celebrity', 'hedge', 'rotted', 'rhinos', 'shuts', 'carried', 'refreshments', 'touchpad', 'vegas']
[VALIDATION] Number total of special tokens: 402
[VALIDATION] Number unique of special tokens: 112
[VALIDATION] Sample

## Embeddings

Decision embedding:
From the three options fasttext, glove and word2vec, seems fasttext to be the most fitting model for the task, because it has the highest chance to recognize the words and their meanings. 
Furthermore in the lecture of SW02 was mentioned that fasttext can hadle unknown words the best. The other two have problems if the word does not appear in their vocabulary. So fasttext was chosen.

Fasttext has multiple possibilities to be used. It was important, to be able to use the word recognition of the model. Gemini suggested to use the model from [fasttext](https://fasttext.cc/docs/en/python-module.html) instead of a shorthand version.


In [14]:
fasttext.util.download_model('en', if_exists='ignore') 
ft_model = fasttext.load_model('cc.en.300.bin')

Create the embedding Matrix from the training vocabulary and the fasttext model vectors.

In [15]:
EMBED_DIM = 300

embedding_matrix = np.zeros((len(vocab), EMBED_DIM))

for word, idx in vocab.items():
    embedding_matrix[idx] = ft_model.get_word_vector(word)

embedding_matrix = torch.tensor(embedding_matrix, dtype=torch.float32)

## First Architecture

**classifier**: the input must match the feature size, use the relu and return a single output. ChatGpt suggested adding dropout to prevent overfitting. It ensures that the calculations are independent.

**pooling**: It was decided to add pooling to recognize the overall meaning , but ChatGpt suggested adding max pooling for strong signals. 

**features**: the goal is to find as many similarities as possible between the goal and the solution. They are similar if their difference is low, when their product and dot product is large, they point to the same direction (cos similarity),  

**forward** calculates the results and return 2 values. Practical for Cross-Entropy validation


In [16]:
class EmbeddingClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=True, padding_idx=0)

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 8 + 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 1)
        )

    def encode(self, x):
        embeddings = self.embedding(x)
        avg_pl = embeddings.mean(dim=1)
        max_pl, _ = embeddings.max(dim=1)
        return torch.cat([avg_pl, max_pl], dim=1)
        

    def compare(self, g, s):
        cos = F.cosine_similarity(g, s, dim=1).unsqueeze(1)
        dot = (g * s).sum(dim=1, keepdim=True)
        return torch.cat([
            g,
            s,
            torch.abs(g - s),
            g * s,
            cos,
            dot
        ], dim=1)

    def forward(self, goal, sol1, sol2):
        g = self.encode(goal)
        s1 = self.encode(sol1)
        s2 = self.encode(sol2)

        feat1 = self.compare(g, s1)
        feat2 = self.compare(g, s2)

        score1 = self.classifier(feat1)
        score2 = self.classifier(feat2)

        return torch.cat([score1, score2], dim=1)

## Second Architecture

Generally it was decided to make the most things as similar as possible to the first architecture, to spot the differences despite similarities.

**rnn**: Gru was chosen, because its simpler and faster. It is bidirectional to recognize the context of the vector better.

**classifier**: The same classifier was used as in the first architecture with the difference, that the feature size is larger for bidirectional rnn.

**pooling**: It was decided wo use tha same pooling as in the first model, but this time it needs a mask to avoid counting zeros as padding.

**features**: The goal is still to find similarities, so the features do not change

**forward**: no change because of the Cross-Entropy Classifier 



In [17]:
class RNNClassifier(nn.Module):
    def __init__(self, embedding_matrix, rnn_hidden_dim=256, 
                 dropout=0.1):
        super().__init__()

        vocab_size, embed_dim = embedding_matrix.shape

        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=False,
            padding_idx=0
        )
    
        self.rnn = nn.GRU(
            input_size=embed_dim,
            hidden_size=rnn_hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        rnn_output_dim = rnn_hidden_dim * 2

        self.classifier = nn.Sequential(
            nn.Linear(rnn_output_dim * 8 + 2, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def get_lengths(self, x):
        return (x != 0).sum(dim=1)

    def encode(self, x):
        max_len = x.size(1)
        lengths = self.get_lengths(x)
        emb = self.embedding(x)
    
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_output, hidden = self.rnn(packed)
    
        output, _ = nn.utils.rnn.pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=max_len
        )
    
        mask = (x != 0).unsqueeze(-1).float()
        output = output * mask
    
        mean_pool = output.sum(dim=1) / (mask.sum(dim=1) + 1e-9)
    
        output_masked = output.masked_fill(mask == 0, float("-inf"))
        max_pool = output_masked.max(dim=1).values
        
        max_pool = torch.where(
            torch.isinf(max_pool),
            torch.zeros_like(max_pool),
            max_pool
        )
    
        return torch.cat([mean_pool, max_pool], dim=1)

    def compare(self, g, s):
        cos = F.cosine_similarity(g, s, dim=1).unsqueeze(1)
        dot = (g * s).sum(dim=1, keepdim=True)

        return torch.cat([
            g,
            s,
            torch.abs(g - s),
            g * s,
            cos,
            dot
        ], dim=1)

    def forward(self, goal, sol1, sol2):
        g = self.encode(goal)
        s1 = self.encode(sol1)
        s2 = self.encode(sol2)

        feat1 = self.compare(g, s1)
        feat2 = self.compare(g, s2)

        score1 = self.classifier(feat1)
        score2 = self.classifier(feat2)

        return torch.cat([score1, score2], dim=1)

## Training

This section includes all the model definitins for the training and parameter tunings for the models.

### Training Workflow

- store the tensors on the gpu
- clear the gradients
- calculate the model's prediction
- calculate loss
- do backpropagation
- optimize the weights
- calculate average loss

In [18]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for goal, sol1, sol2, label in loader:
        goal, sol1, sol2, label = goal.to(DEVICE), sol1.to(DEVICE), sol2.to(DEVICE), label.to(DEVICE)

        optimizer.zero_grad()
        output = model(goal, sol1, sol2)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

### Evaluation Workflow

- stop storing backpropagation information
- store the tensors on the gpu
- calculate the model's prediction
- calculate loss
- compare prediction with solution
- calculate accuracy and average loss

In [19]:
def evaluate(model, loader, criterion):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0

    with torch.no_grad():
        for goal, sol1, sol2, label in loader:
            goal, sol1, sol2, label = goal.to(DEVICE), sol1.to(DEVICE), sol2.to(DEVICE), label.to(DEVICE)

            output = model(goal, sol1, sol2)
            loss = criterion(output, label)
            total_loss += loss.item()
            preds = output.argmax(dim=1)
            correct += (preds == label).sum().item()
            total += label.size(0)

    return correct / total, total_loss / len(loader)

### Early stopping
To prevent the overfitting of the model, early stopping was added especially for the rnn.
After the stop, the restoreation of the best model state is necessary for the evaluation.

In [20]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1

        return self.counter >= self.patience

    def restore(self, model):
        model.load_state_dict(self.best_state)

### Save Checkpoint

The best model state should not be lost, so a local chackpoint can save it as backup.

In [21]:
def save_checkpoint(model, optimizer, epoch, val_loss, path="best.pt"):
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "epoch": epoch,
        "val_loss": val_loss
    }, path)

### First Experiment Tracking

Regarding the experiment tracking, there were four decisions to make:

**parameter to be tuned**: Originally there are five parameters to be tuned: 
- learning_rate
- batch_size
- hidden_dim
- dropout
- weight_decay (potentially for adamW)

**parameter values**: ChatGpt suggested the most reasonable potential values for the parameters. 

**tuning method**: The goal is to find the best parameter combination with values. The best way is to use the grid method where there is a run for each parameter combination. With five parameters having at least two values, it would require 32 runs to have all combinations. So ChatGpt prioritized the five items and learning_rate, batch_size and hidden_dim were the top three. These were chosen, requiring 12 runs.  

**attributes for the log**: The four measurements, which can help to see how the training fits the model: train_acc, train_loss, val_acc, val_loss.

### Loss function

During the course oneloss functions was covered mre detailed (SW03): Cross-Entropy
It is useful for 2 Class classification, predicting the likeliness of the 2 classes. Perfect for the use-case, it is used for both architectures.

###  Epochs
Epochs are helpful to recognize some patterns during the training. Ten is not too much, but enough to see patterns.

### Optimizer

The two most popular choices for Optimizers are adam and adamW according to Gemini. Furthermore adam was used during the course (SW01). Since finetuning is a step in this project and weight decay is among not the more important parameters, the choice is on adam.

###


In [91]:
def run_experiment1(config=None):
    with wandb.init(config=config, entity="nlp-1"):
        config = wandb.config

        train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=config.batch_size)

        model = EmbeddingClassifier(embedding_matrix, hidden_dim=config.hidden_dim).to(DEVICE)

        optimizer = optim.Adam(
            model.parameters(),
            lr=config.learning_rate,
        ) 
        criterion = nn.CrossEntropyLoss()
        
        early_stopper = EarlyStopping(patience=5)
        best_val_loss = float("inf")
        
        for epoch in range(config.epochs):
            train_loss = train(model, train_loader, optimizer, criterion)
            train_acc, _ = evaluate(model, train_loader, criterion)
            val_acc, val_loss = evaluate(model, val_loader, criterion)

            wandb.log({
                "train_acc": train_acc,
                "train_loss": train_loss,
                "val_acc": val_acc,
                "val_loss": val_loss
            })

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                save_checkpoint(model, optimizer, epoch, val_loss, "model1.pt")

            if early_stopper.step(val_loss, model):
                print("Early stopping triggered")
                break
                
        early_stopper.restore(model)
         

sweep_config = {
    "method": "grid",
    "metric": {
        "name": "val_acc",
        "goal": "maximize"
    },
    "parameters": {
        "learning_rate": {"values": [1e-5, 1e-4, 1e-3]},
        "hidden_dim": {"values": [128, 256]},
        "batch_size": {
            "values": [32, 64]
        },
        "epochs": {
            "value": 10
        }
    }
}

In [92]:
sweep_id = wandb.sweep(sweep_config, project="nlp-project1")
wandb.agent(sweep_id, function=run_experiment1)

Create sweep with ID: 0zpoxbic
Sweep URL: https://wandb.ai/nlp-1/nlp-project1/sweeps/0zpoxbic


wandb: Agent Starting Run: z2ckc4bw with config:
wandb: 	batch_size: 32
wandb: 	epochs: 10
wandb: 	hidden_dim: 128
wandb: 	learning_rate: 1e-05
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jovyan/.netrc.


train_acc,▁▃▄▆▇▇▇███
train_loss,██▇▅▆▄▄▂▂▁
val_acc,▁▂▃▅▆▆███▇
val_loss,█▇▆▆▅▄▃▂▂▁
train_acc,0.55945
train_loss,0.6914
val_acc,0.533
val_loss,0.69195


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: fwi0lc7l with config:
wandb: 	batch_size: 32
wandb: 	epochs: 10
wandb: 	hidden_dim: 128
wandb: 	learning_rate: 0.0001
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jovyan/.netrc.


train_acc,▁▄▅▆▆▇▇███
train_loss,█▇▇▆▅▅▃▃▂▁
val_acc,▁▅▅█▆▆▅▇█▄
val_loss,█▇▆▅▄▄▃▂▁▁
train_acc,0.5799
train_loss,0.68152
val_acc,0.534
val_loss,0.68689


...

wandb: Agent Starting Run: 1rlm5e2w with config:
wandb: 	batch_size: 64
wandb: 	epochs: 10
wandb: 	hidden_dim: 256
wandb: 	learning_rate: 0.0001
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jovyan/.netrc.


train_acc,▁▅▆▆▆▇▇███
train_loss,█▇▇▆▅▄▄▃▂▁
val_acc,▄█▆▆▆▁▅▁▁▂
val_loss,█▇▆▅▄▃▃▂▂▁
train_acc,0.58063
train_loss,0.68114
val_acc,0.537
val_loss,0.68693


wandb: Agent Starting Run: zckwbqmo with config:
wandb: 	batch_size: 64
wandb: 	epochs: 10
wandb: 	hidden_dim: 256
wandb: 	learning_rate: 0.001
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jovyan/.netrc.


train_acc,▁▂▃▄▅▆▆▆▇█
train_loss,█▇▆▅▄▄▃▁▁▁
val_acc,▃▁▃▂▄▅▄▅▆█
val_loss,█▇▆▄▃▃▃▂▂▁
train_acc,0.5973
train_loss,0.67684
val_acc,0.563
val_loss,0.68427


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


### Second Experiment Tracking

Regarding the parmeter tuning, there were same decisions to make:

**parameter to be tuned**: This time were some more parameters to be tuned: 
- learning_rate
- batch_size
- hidden_dim
- dropout
- weight_decay (for adamW)
- rnn_hidden_dim

**parameter values**: ChatGpt suggested the most reasonable potential values for the parameters. 

**tuning method**: Same as last time, the goal is to find the best parameter combination with values. So for the best combination, the grid method will be used with a total of 12 runs.  

**attributes for the log**: The same four as last time, can help to interpret the same things.

### Loss function & Epochs
Both of them are unchanged.


### Optimizer
This time was AdamW chosen since weight_decay has a higher importance in this model.

In [106]:
def run_experiment2(config=None):
    with wandb.init(config=config, entity="nlp-1"):
        config = wandb.config

        model2 = RNNClassifier(
            embedding_matrix,
            rnn_hidden_dim=config.rnn_hidden_dim,
            dropout=config.dropout,
        ).to(DEVICE)

        optimizer = optim.AdamW(
            model2.parameters(),
            lr=1e-4,
            weight_decay=config.weight_decay
        )
            
        criterion = nn.CrossEntropyLoss()
        early_stopper = EarlyStopping(patience=5)
        best_val_loss = float("inf")

        for epoch in range(config.epochs):
            train_loss = train(model2, train_loader, optimizer, criterion)
            train_acc, _ = evaluate(model2, train_loader, criterion)
            val_acc, val_loss = evaluate(model2, val_loader, criterion)

            wandb.log({
                "train_acc": train_acc,
                "train_loss": train_loss,
                "val_acc": val_acc,
                "val_loss": val_loss
            })
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                save_checkpoint(model2, optimizer, epoch, val_loss, "model2.pt")
        
            if early_stopper.step(val_loss, model2):
                print("Early stopping triggered")
                break
                
        early_stopper.restore(model2)

sweep_config2 = {
    "method": "grid",
    "metric": {
        "name": "val_acc",
        "goal": "maximize"
    },
    "parameters": {
        "rnn_hidden_dim": {"values": [256, 512]},
        "dropout": {"values": [0.1, 0.3]},
        "weight_decay": {"values": [0.0, 1e-5, 1e-4]},
        "epochs": {
            "value": 10
        }
    }
}

In [107]:
sweep_id2 = wandb.sweep(sweep_config2, project="nlp-project1-2")
wandb.agent(sweep_id2, function=run_experiment2)

Create sweep with ID: 9jolg3vh
Sweep URL: https://wandb.ai/nlp-1/nlp-project1-2/sweeps/9jolg3vh


wandb: Agent Starting Run: bf187zpv with config:
wandb: 	dropout: 0.1
wandb: 	epochs: 10
wandb: 	rnn_hidden_dim: 256
wandb: 	weight_decay: 0
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jovyan/.netrc.


Early stopping triggered


train_acc,▁▃▅▆▇▇█
train_loss,██▆▅▃▂▁
val_acc,▁▅▆▆▇▇█
val_loss,▂▁▁▂▃▆█
train_acc,0.82836
train_loss,0.40901
val_acc,0.599
val_loss,0.84218


wandb: Agent Starting Run: gkzis88a with config:
wandb: 	dropout: 0.1
wandb: 	epochs: 10
wandb: 	rnn_hidden_dim: 256
wandb: 	weight_decay: 1e-05
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jovyan/.netrc.


...

train_acc,▁▃▅▆▇▇█
train_loss,██▆▅▃▂▁
val_acc,▁▂▆▆█▇█
val_loss,▂▁▁▁▃▄█
train_acc,0.82684
train_loss,0.43212
val_acc,0.588
val_loss,0.8429


wandb: Agent Starting Run: o304v6rt with config:
wandb: 	dropout: 0.3
wandb: 	epochs: 10
wandb: 	rnn_hidden_dim: 512
wandb: 	weight_decay: 0.0001
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jovyan/.netrc.


Early stopping triggered


train_acc,▁▃▅▆▇▇█
train_loss,██▆▅▃▂▁
val_acc,▁▅▇▅▇██
val_loss,▂▁▁▂▂▆█
train_acc,0.8236
train_loss,0.42684
val_acc,0.587
val_loss,0.84539


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


 ## Evaluation

For the tests all the hyperparameter result from the experiment tracking were adjusted and one more training is started with them.

##### **First Architecture Parameters**
They were chosen from the run soft-sweep-12 for having the highest train_acc, val_acc and highest train_loss and val_loss decrease
batch_size=64, learning_rate=1e-3, hidden_dim=256




In [23]:
criterion = nn.CrossEntropyLoss()
EPOCHS = 10

In [80]:
early_stopper = EarlyStopping(patience=5)
best_val_loss = float("inf")

model1 = EmbeddingClassifier(embedding_matrix, hidden_dim=256).to(DEVICE) 
opt1 = optim.Adam(model1.parameters(), lr=1e-3)

for epoch in range(EPOCHS):
    train_loss = train(model1, train_loader, opt1, criterion)
    train_acc, _ = evaluate(model1, train_loader, criterion)
    val_acc, val_loss = evaluate(model1, val_loader, criterion)

    print(f"[Classifier] Epoch {epoch+1}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model1, opt1, epoch, val_loss, "model1.pt")

    if early_stopper.step(val_loss, model1):
        print("Early stopping triggered")
        break

early_stopper.restore(model1)

[Classifier] Epoch 1: Train Loss=0.6932, Train Acc=0.5720, Val Loss=0.6912, Val Acc=0.5380
[Classifier] Epoch 2: Train Loss=0.6906, Train Acc=0.5706, Val Loss=0.6896, Val Acc=0.5400
[Classifier] Epoch 3: Train Loss=0.6880, Train Acc=0.5762, Val Loss=0.6881, Val Acc=0.5400
[Classifier] Epoch 4: Train Loss=0.6857, Train Acc=0.5789, Val Loss=0.6869, Val Acc=0.5390
[Classifier] Epoch 5: Train Loss=0.6839, Train Acc=0.5842, Val Loss=0.6863, Val Acc=0.5420
[Classifier] Epoch 6: Train Loss=0.6814, Train Acc=0.5876, Val Loss=0.6862, Val Acc=0.5360
[Classifier] Epoch 7: Train Loss=0.6803, Train Acc=0.5933, Val Loss=0.6855, Val Acc=0.5420
[Classifier] Epoch 8: Train Loss=0.6794, Train Acc=0.5937, Val Loss=0.6850, Val Acc=0.5470
[Classifier] Epoch 9: Train Loss=0.6769, Train Acc=0.5979, Val Loss=0.6836, Val Acc=0.5520
[Classifier] Epoch 10: Train Loss=0.6750, Train Acc=0.5944, Val Loss=0.6844, Val Acc=0.5410


In [45]:
test_acc1, test_val1 = evaluate(model1, test_loader, criterion)
print(f"\nFinal Test Accuracy: {test_acc1:.4f}")


Final Test Accuracy: 0.5631


##### **Second Architecture Parameters**

There were two best results amoung the configurations helpful-sweep-6 and misty-sweep-5. For the test was helpful-sweep-6 used: rnn_hidden_dim=512, dropout=0.1 and weight_decay=1e-4

In [25]:
early_stopper2 = EarlyStopping(patience=5)
best_val_loss2 = float("inf")

model2 = RNNClassifier(
    embedding_matrix,
    rnn_hidden_dim=512,
    dropout=0.1
).to(DEVICE)
opt2 = optim.AdamW(model2.parameters(), lr=1e-4, weight_decay=1e-4)

for epoch in range(EPOCHS):
    train_loss = train(model2, train_loader, opt2, criterion)
    train_acc, _ = evaluate(model2, train_loader, criterion)
    val_acc, val_loss = evaluate(model2, val_loader, criterion)
    print(f"[RNN] Epoch {epoch+1}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

    if val_loss < best_val_loss2:
        best_val_loss2 = val_loss
        save_checkpoint(model2, opt2, epoch, val_loss, "model2.pt")

    if early_stopper2.step(val_loss, model2):
        print("Early stopping triggered")
        break

early_stopper2.restore(model2)

[RNN] Epoch 1: Train Loss=0.6922, Train Acc=0.5903, Val Loss=0.6837, Val Acc=0.5410
[RNN] Epoch 2: Train Loss=0.6649, Train Acc=0.6772, Val Loss=0.6798, Val Acc=0.5700
[RNN] Epoch 3: Train Loss=0.6030, Train Acc=0.7318, Val Loss=0.6978, Val Acc=0.5840
[RNN] Epoch 4: Train Loss=0.5399, Train Acc=0.7673, Val Loss=0.7029, Val Acc=0.5870
[RNN] Epoch 5: Train Loss=0.4865, Train Acc=0.8004, Val Loss=0.7150, Val Acc=0.5910
[RNN] Epoch 6: Train Loss=0.4325, Train Acc=0.8272, Val Loss=0.7830, Val Acc=0.5970
[RNN] Epoch 7: Train Loss=0.3835, Train Acc=0.8517, Val Loss=0.8422, Val Acc=0.6010
Early stopping triggered


In [26]:
test_acc2, test_val2 = evaluate(model2, test_loader, criterion)
print(f"\nFinal Test Accuracy: {test_acc2:.4f}")


Final Test Accuracy: 0.5958


## Interpretation

The interpretations are mainly based on the wandb results.
[wandb Report](https://wandb.ai/nlp-1/nlp-project1/reports/NLP-Project-one-report--VmlldzoxNjQ0MDQxMQ)

##### **First Architecture**
In the wandb charts in each run both train_loss and val_loss decrease consistently. The train_acc increases consistently whereas the val_acc inconsistently with some noise. These signs show (especially the consistent ones), that the model is learning something and improving itself. However the increase/decrease rate is small, so the model cannot predict reliably with a test accuracy of 56.31%. The chance, that the model predicts the answer correctly is still slightly better then choosing a random answer.

##### **Second Architecture**
At first look the training of the model seamed to go well, since train_loss is consistently decrease and train_acc the same way increase. Even the validation seems to work correctly for the first few epochs, but it always starts to increase, which leads to overfitting. Thanks to early stopping, it is able to get a test accuracy of 59.58%.

##### **Comparison**
Despite the models having many similarities, they have completely different results. For unseen data the second architecture would work slightly better with a short effective learning-curve (val_loss). However the parameter tuning worked for the first architecture better, because the best configuration was clearly better then the others. In the second architecture there was no configuration having all the metrics better then the others.

